# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedm0ssad/Machine-Learning-Intern/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

My lane: **Refresh / Content Opportunity Scoring** — ranking which pages an editor should review first for refresh, expansion, protection, or pruning.

Why this one: it builds directly on what I measured in ML-01, where a learned ranking reached ~2.8x the hand-written rule's Precision@50 on a client-holdout split. It is also the lane whose output a real actor uses — a ranked review queue with reason codes — rather than a report nobody acts on. And it points forward: the starter's `is_declining_label` is computed from the same 90-day window as its features, so it is only a proxy target; weeks 3+ unlock the warehouse daily history this lane needs to define a label from a *future* window (features from a prior 90 days -> decline measured in the next 30 days).

In [ ]:
# Data anchor: the starter slice itself backs this lane - one row per page, enough volume to rank.
from pathlib import Path

candidates = [
    Path("../../data/raw/content_refresh_anonymized.csv"),  # run from work/notebooks (nbconvert default)
    Path("data/raw/content_refresh_anonymized.csv"),        # run from the repo root
]
csv_path = next(p for p in candidates if p.exists())

import pandas as pd

df = pd.read_csv(csv_path)
print(f"rows x cols: {df.shape[0]:,} x {df.shape[1]} | distinct clients: {df['client_id'].nunique()}")

rows x cols: 30,000 x 44 | distinct clients: 32


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

- **Decision improved:** which pages an editor reviews first. Not "predict decline" — an editor has limited weekly capacity and must choose an *order*; a ranked queue improves that decision.
- **Who acts on it:** the content editor (or a client team's reviewer) who works through the top of the queue each week. If nobody would act on the output, the work has no customer.
- **Cost of a wrong call — asymmetric and capacity-bound.** A false alarm costs a few editor hours spent on a page that did not need it; a miss lets a high-demand page keep bleeding impressions and clicks. Because traffic is concentrated in a small head of pages (section 3), a miss up there hurts far more than a miss deep in the tail.
- **Metric named before training:** since only the top K pages get worked, "good" must be measured as **Precision@K** (of the top K surfaced, how many truly deserved review), not accuracy.

In [ ]:
# Capacity math: the naive queue (all visible declining pages) is far beyond one editor's weekly capacity.
# impressions_90d >= 500 = the lane guide's "enough demand to matter" floor.
# trend_direction is the label side ONLY - it is never a feature (data-dictionary rule).

m_visible = df["impressions_90d"] >= 500
declining = df["trend_direction"] == "down"

n_visible = int(m_visible.sum())
n_visible_declining = int((m_visible & declining).sum())
weeks_of_queue = n_visible_declining / 50  # ~50 reviews per week

print(f"visible pages (impressions_90d >= 500): {n_visible:,}")
print(f"...of them declining: {n_visible_declining:,} ({declining[m_visible].mean() * 100:.1f}% of visible)")
print(f"at ~50 reviews/week, that queue alone is ~{weeks_of_queue:.0f} weeks -> ranking 'which first' is unavoidable")

visible pages (impressions_90d >= 500): 16,726
...of them declining: 9,961 (59.6% of visible)
at ~50 reviews/week, that queue alone is ~199 weeks -> ranking 'which first' is unavoidable


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers that justify seven weeks on this lane:

1. **54.2% of pages are declining.** "Find me declining pages" is nearly a coin flip in this slice — the hard part is not detection, it is *ordering* pages so review capacity is spent where misses hurt most.
2. **The top 10% of pages hold 80.5% of all clicks.** The cost of errors concentrates in a small head of the inventory — that is exactly why ranking quality at the top (Precision@K) is the metric that matters, not model-wide accuracy.
3. **9,961 visible pages (impressions_90d >= 500) are already declining.** That is a naive first queue — at ~50 reviews/week it is ~200 weeks of work, far beyond any editor's capacity. The binding constraint is review capacity, so the queue must say *which first*.

In [ ]:
# The three numbers, computed live from the starter CSV.
n = len(df)

down_share = (df["trend_direction"] == "down").mean() * 100
top10 = df.sort_values("clicks_90d", ascending=False).head(n // 10)
click_share = top10["clicks_90d"].sum() / df["clicks_90d"].sum() * 100
m_visible = df["impressions_90d"] >= 500
queue = int((m_visible & (df["trend_direction"] == "down")).sum())

print(f"1) declining share of pages: {down_share:.1f}%")
print(f"2) share of all clicks_90d held by the top 10% of pages: {click_share:.1f}%")
print(f"3) visible pages (impressions_90d >= 500) that are declining: {queue:,}")

1) declining share of pages: 54.2%
2) share of all clicks_90d held by the top 10% of pages: 80.5%
3) visible pages (impressions_90d >= 500) that are declining: 9,961


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim (careful words only):**

- **Observed / measured:** factual counts and comparisons computed from this slice (e.g. "54.2% of pages show declining last-30d impressions"), stated only for this 30k anonymized slice.
- **Directional:** ML-01's client-holdout comparison — a learned ranking scored higher Precision@20/@50 than the hand rule; suggestive, and stated as one run, not a guarantee.
- **Decision-support:** the queue's job is ranking *for a human reviewer* — "these pages first, for review". The reviewer decides; nothing here says a refresh fixes anything.

**What I will never claim:**

- **Not causal:** no "refreshing this page caused a recovery" — there is no experiment in the data, and no causal design is available to me.
- **Not "predicting Google":** no claim about any factor of Google's algorithm; the data is observational only.
- **Not the starter label as truth:** `is_declining_label` derives from `trend_direction` / `trend_pct`, both computed on the same window as the features — so any starter-window result stays a proxy finding. Those columns are label-side and will never be features.

In [ ]:
# Label-side reality check (counts only - these columns are the label family, so they are never features).
# trend_pct is undefined when prev-30d impressions are 0 - the data dictionary itself says the label has holes.

print("trend_direction counts:", df["trend_direction"].value_counts().to_dict())
print("trend_pct rows blank (undefined when impressions_prev_30d = 0):", int(df["trend_pct"].isna().sum()))

trend_direction counts: {'down': 16262, 'stable': 5962, 'up': 4388, 'new': 2236, 'flat': 1152}
trend_pct rows blank (undefined when impressions_prev_30d = 0): 3388
